In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets/sample_submission.csv
/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets/train.csv
/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets/test.csv


In [2]:
#Score: 0.91891
#Public score: 0.91948
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import gc

print("="*90)
print("🚀 CORRECT SUBMISSION - FIXED PATHS")
print("="*90)

# 1. ЗАГРУЖАЕМ ПРАВИЛЬНЫЕ ДАННЫЕ С КОРРЕКТНЫМИ ПУТЯМИ
print("\n📥 Loading correct data...")

# Тренировочные данные
train_path = '/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets/train.csv'
test_path = '/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets/test.csv'

# Загружаем train (8.2M) с семплированием 30%
print("Loading training data...")
train_df = pd.read_csv(train_path, low_memory=False)
train_df = train_df.sample(frac=0.3, random_state=42)  # 2.47M строк
train_df['app_service'] = train_df['app_service'].astype(str)
print(f"✅ Train shape: {train_df.shape}")

# Загружаем test (4.7M) - полностью
print("Loading test data...")
test_df = pd.read_csv(test_path, low_memory=False)
print(f"✅ Test shape: {test_df.shape}")  # Должно быть 4,730,370

# 2. БЫСТРОЕ ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ
print("\n⚙️ Extracting features (vectorized)...")

tcp_cols = [f'tcp_len_{i}' for i in range(1, 31)]

def extract_features_vectorized(df):
    """Быстрое извлечение признаков с numpy"""
    packets = df[tcp_cols].values.astype(np.int16)
    
    features = {
        'out_cnt': np.sum(packets > 0, axis=1),
        'in_cnt': np.sum(packets < 0, axis=1),
        'out_vol': np.sum(np.maximum(packets, 0), axis=1),
        'in_vol': -np.sum(np.minimum(packets, 0), axis=1),
        'mtu_out': np.sum(packets > 1200, axis=1),
        'mtu_in': np.sum(packets < -1200, axis=1),
        'max_out': np.max(np.where(packets > 0, packets, 0), axis=1),
        'max_in': -np.min(np.where(packets < 0, packets, 0), axis=1),
    }
    
    # Первые 10 пакетов
    for i in range(10):
        features[f'p{i+1}'] = packets[:, i]
    
    # Направления первых 3 пакетов
    features['dir1'] = np.sign(packets[:, 0])
    features['dir2'] = np.sign(packets[:, 1])
    features['dir3'] = np.sign(packets[:, 2])
    
    # Смена направлений
    directions = np.sign(packets)
    features['dir_changes'] = np.sum(np.abs(np.diff(directions, axis=1)) > 0, axis=1)
    
    return pd.DataFrame(features, dtype=np.float32)

# Извлекаем признаки
print("Extracting train features...")
X_train = extract_features_vectorized(train_df)
y_train = train_df['app_service'].values

print("Extracting test features...")
X_test = extract_features_vectorized(test_df)

print(f"✅ Train features: {X_train.shape}")
print(f"✅ Test features: {X_test.shape}")
print(f"✅ Memory usage: {X_train.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# 3. КОДИРУЕМ ЦЕЛИ
print("\n🏷️ Encoding labels...")
le = LabelEncoder()
y_enc = le.fit_transform(y_train)
print(f"✅ Classes: {len(le.classes_)}")

# 4. ОБУЧАЕМ МОДЕЛЬ
print("\n🚀 Training LightGBM...")

params = {
    'objective': 'multiclass',
    'num_class': len(le.classes_),
    'metric': 'multi_logloss',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_lambda': 1.0,
    'reg_alpha': 0.5,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

model = lgb.LGBMClassifier(**params)
model.fit(X_train, y_enc)

# 5. ПРЕДСКАЗАНИЯ (быстро)
print("\n🎯 Making predictions...")
test_pred = model.predict(X_test)
test_pred_labels = le.inverse_transform(test_pred)

# 6. СОЗДАЕМ САБМИТ
print("\n📝 Creating submission...")
submission = pd.DataFrame({
    'id': test_df['id'],
    'app_service': test_pred_labels
})

# Проверяем размер
print(f"✅ Submission shape: {submission.shape}")
print(f"✅ Expected: (4730370, 2)")

if len(submission) == 4730370:
    print("✅ Perfect! Correct number of rows!")
else:
    print(f"⚠️ Warning: Got {len(submission)} rows, expected 4730370")

# Сохраняем
submission.to_csv('submission_fixed.csv', index=False)

print("="*90)
print("✅ DONE!")
print(f"Submission saved: submission_fixed.csv")
print(f"Rows: {len(submission):,}")
print(f"Classes predicted: {submission['app_service'].nunique()}")
print("="*90)

# Показываем распределение предсказаний
print("\n📊 Top 10 predicted classes:")
print(submission['app_service'].value_counts().head(10))

🚀 CORRECT SUBMISSION - FIXED PATHS

📥 Loading correct data...
Loading training data...
✅ Train shape: (2474564, 31)
Loading test data...
✅ Test shape: (4730370, 31)

⚙️ Extracting features (vectorized)...
Extracting train features...
Extracting test features...
✅ Train features: (2474564, 22)
✅ Test features: (4730370, 22)
✅ Memory usage: 0.20 GB

🏷️ Encoding labels...
✅ Classes: 252

🚀 Training LightGBM...

🎯 Making predictions...

📝 Creating submission...
✅ Submission shape: (4730370, 2)
✅ Expected: (4730370, 2)
✅ Perfect! Correct number of rows!
✅ DONE!
Submission saved: submission_fixed.csv
Rows: 4,730,370
Classes predicted: 252

📊 Top 10 predicted classes:
app_service
343             107729
bing            103003
Apple           102966
287             101780
120             100925
mail_ru         100028
tria             99986
Telegram         99922
1                99885
apple_icloud     99769
Name: count, dtype: int64


In [3]:
#Score: 0.83238
#Public score: 0.83278
# ============================================================================
# 🚀 TCP TRAFFIC - SOLUTION WITH ALL CLASSES
# ============================================================================

import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "lightgbm", "imbalanced-learn", "-q"])

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import xgboost as xgb
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*90)
print("🚀 TCP TRAFFIC - ALL CLASSES SOLUTION")
print("="*90)

# 1. ЗАГРУЗКА
print("\n📥 Загрузка данных...")
import kagglehub
path = kagglehub.competition_download('whos-talking-classify-the-app-by-its-packets')
train_path = f"{path}/train.csv"
test_path = f"{path}/test.csv"

# Загружаем все данные, чтобы увидеть все классы
train_df = pd.read_csv(train_path, low_memory=False)
train_df['app_service'] = train_df['app_service'].astype(str)
print(f"✅ Train shape: {train_df.shape}")
print(f"✅ Всего классов: {train_df['app_service'].nunique()}")

# 2. АНАЛИЗ РАСПРЕДЕЛЕНИЯ КЛАССОВ
class_counts = train_df['app_service'].value_counts()
rare_classes = class_counts[class_counts < 1000].index.tolist()
freq_classes = class_counts[class_counts >= 1000].index.tolist()
print(f"✅ Частых классов (>1000): {len(freq_classes)}")
print(f"✅ Редких классов (<1000): {len(rare_classes)}")

# 3. СБАЛАНСИРОВАННАЯ ВЫБОРКА
print("\n⚙️ Создание сбалансированной выборки...")

# Для частых классов берем по 2000 samples
freq_samples = []
for cls in freq_classes:
    cls_data = train_df[train_df['app_service'] == cls]
    if len(cls_data) > 2000:
        freq_samples.append(cls_data.sample(n=2000, random_state=42))
    else:
        freq_samples.append(cls_data)

# Для редких классов берем все
rare_samples = [train_df[train_df['app_service'].isin(rare_classes)]]

# Объединяем
balanced_df = pd.concat(freq_samples + rare_samples, ignore_index=True)
print(f"✅ Сбалансированный датасет: {balanced_df.shape}")
print(f"✅ Классов в обучении: {balanced_df['app_service'].nunique()}")

# 4. ПРИЗНАКИ
def extract_features(df):
    features = pd.DataFrame(index=df.index)
    tcp_data = df[[f'tcp_len_{i}' for i in range(1, 31)]].values.astype(np.float32)
    
    outgoing = np.where(tcp_data > 0, tcp_data, 0)
    incoming = np.where(tcp_data < 0, -tcp_data, 0)
    directions = np.sign(tcp_data)
    
    eps = 1e-7
    
    # Топ-20 признаков
    features['out_cnt'] = np.sum(outgoing > 0, axis=1)
    features['in_cnt'] = np.sum(incoming > 0, axis=1)
    features['out_vol'] = np.sum(outgoing, axis=1)
    features['in_vol'] = np.sum(incoming, axis=1)
    features['ratio_vol'] = features['out_vol'] / (features['in_vol'] + eps)
    
    features['mtu_out'] = np.sum(outgoing > 1200, axis=1)
    features['mtu_in'] = np.sum(incoming > 1200, axis=1)
    
    for i in range(10):
        features[f'p{i+1}'] = tcp_data[:, i]
        features[f'dir{i+1}'] = directions[:, i]
    
    features['mean_out'] = features['out_vol'] / (features['out_cnt'] + eps)
    features['mean_in'] = features['in_vol'] / (features['in_cnt'] + eps)
    features['dir_changes'] = np.sum(np.abs(np.diff(directions, axis=1)) > 0, axis=1)
    
    return features.fillna(0).astype(np.float32)

# 5. ПОДГОТОВКА
print("\n⚙️ Извлечение признаков...")
X = extract_features(balanced_df)
y = balanced_df['app_service'].values
print(f"✅ X shape: {X.shape}")

le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. ТЕСТ
test_df = pd.read_csv(test_path, low_memory=False)
X_test = extract_features(test_df)
X_test_scaled = scaler.transform(X_test)

# 7. КРОСС-ВАЛИДАЦИЯ С УЧЕТОМ ВСЕХ КЛАССОВ
print("\n🚀 Обучение с кросс-валидацией...")

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
test_predictions = np.zeros((len(X_test_scaled), len(le.classes_)))
val_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y_enc)):
    print(f"\n🎯 Fold {fold + 1}/{n_folds}")
    
    X_fold_train, X_fold_val = X_scaled[train_idx], X_scaled[val_idx]
    y_fold_train, y_fold_val = y_enc[train_idx], y_enc[val_idx]
    
    dtrain = xgb.DMatrix(X_fold_train, label=y_fold_train)
    dval = xgb.DMatrix(X_fold_val, label=y_fold_val)
    
    params = {
        'objective': 'multi:softprob',
        'num_class': len(le.classes_),
        'max_depth': 8,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_lambda': 2,
        'reg_alpha': 1,
        'eval_metric': 'mlogloss',
        'seed': 42
    }
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=300,
        evals=[(dval, 'val')],
        early_stopping_rounds=30,
        verbose_eval=50
    )
    
    # Валидация
    val_pred = model.predict(dval)
    val_pred_labels = np.argmax(val_pred, axis=1)
    val_f1 = f1_score(y_fold_val, val_pred_labels, average='macro')
    val_scores.append(val_f1)
    print(f"   Fold F1: {val_f1:.4f}")
    
    # Тест
    dtest = xgb.DMatrix(X_test_scaled)
    test_pred = model.predict(dtest)
    test_predictions += test_pred / n_folds

print(f"\n✅ Средний валидационный F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")

# 8. ФИНАЛЬНОЕ ПРЕДСКАЗАНИЕ
test_pred_labels = np.argmax(test_predictions, axis=1)
test_preds = le.inverse_transform(test_pred_labels)

# 9. СОХРАНЕНИЕ
submission = pd.DataFrame({
    'id': range(len(test_preds)),
    'app_service': test_preds
})
submission.to_csv('all_classes_solution.csv', index=False)

print("="*90)
print("✅ ГОТОВО!")
print("="*90)
print(f"\n📊 Статистика:")
print(f"   • Предсказано файлов: {len(test_preds)}")
print(f"   • Уникальных классов: {submission['app_service'].nunique()}")
print(f"   • Средний валидационный F1: {np.mean(val_scores):.4f}")

🚀 TCP TRAFFIC - ALL CLASSES SOLUTION

📥 Загрузка данных...
✅ Train shape: (8248546, 31)
✅ Всего классов: 252
✅ Частых классов (>1000): 252
✅ Редких классов (<1000): 0

⚙️ Создание сбалансированной выборки...
✅ Сбалансированный датасет: (477175, 31)
✅ Классов в обучении: 252

⚙️ Извлечение признаков...
✅ X shape: (477175, 30)

🚀 Обучение с кросс-валидацией...

🎯 Fold 1/5
[0]	val-mlogloss:3.40888
[50]	val-mlogloss:0.67162
[100]	val-mlogloss:0.39954
[150]	val-mlogloss:0.31720
[200]	val-mlogloss:0.27998
[250]	val-mlogloss:0.25994
[299]	val-mlogloss:0.24870
   Fold F1: 0.9317
